# File Handling — Saving Information After Python Stops 📁

## Goal

A file is a named container stored on disk. Text mode converts bytes using an encoding; binary mode gives raw bytes. Destructive demonstrations use temporary copies so the existing `example.txt`, `example.bin`, and `destination.txt` remain safe.

## 1. `open`, modes, and closing

| Mode | Job | Missing file | Existing file |
|---|---|---|---|
| `r` | read | error | preserve |
| `w` | write | create | **erase first** |
| `a` | append | create | preserve/add end |
| `x` | exclusive create | create | error |
| `+` | add read/write ability | depends | depends |
| `b` / `t` | binary / text | modifier | modifier |

`with open(...) as file:` closes automatically, even after an exception. Prefer it to manual `close()`.

In [1]:
from pathlib import Path
path=Path('example.txt')
with path.open('r',encoding='utf-8') as file:
    content=file.read()
    print(content)
print('closed:',file.closed)

Hello world
This is a new line 

closed: True


## 2. `read`, `readline`, `readlines`, iteration

`read(size)` gets text/chars, `readline()` one line, `readlines()` a list of remaining lines. Direct iteration streams one line at a time and suits large files.

In [2]:
with open('example.txt',encoding='utf-8') as file:
    print('first 10:',repr(file.read(10)))
    print('next line:',repr(file.readline()))
    print('remaining:',file.readlines())
with open('example.txt',encoding='utf-8') as file:
    for number,line in enumerate(file,1): print(number,line.rstrip('\n'))

first 10: 'Hello worl'
next line: 'd\n'
remaining: ['This is a new line \n']
1 Hello world
2 This is a new line 


## 3. Write, append, and `writelines`

`write` returns the number of characters. `writelines` does **not** insert newlines. The original overwrite/append examples run inside a disposable directory.

In [3]:
import tempfile
with tempfile.TemporaryDirectory() as folder:
    demo=Path(folder)/'example.txt'
    with demo.open('w',encoding='utf-8') as file:
        file.write('Hello World!\n'); file.write('This is a new line.\n')
    with demo.open('a',encoding='utf-8') as file:
        file.write('Append operation taking place!\n')
        file.writelines(['First line\n','Second line\n','Third line\n'])
    print(demo.read_text(encoding='utf-8'))

Hello World!
This is a new line.
Append operation taking place!
First line
Second line
Third line



## 4. File pointer: `tell` and `seek`

The pointer is the reading/writing bookmark. `tell()` reports its position; `seek()` moves it. After writing in `w+`, seek to zero before reading.

In [4]:
with tempfile.TemporaryDirectory() as folder:
    demo=Path(folder)/'pointer.txt'
    with demo.open('w+',encoding='utf-8') as file:
        file.write('Hello world\nThis is a new line\n')
        print('after write:',file.tell())
        file.seek(0)
        print('after seek:',file.tell())
        print(file.read())

after write: 33
after seek: 0
Hello world
This is a new line



## 5. Binary files

Binary mode uses `bytes`, not strings, and has no text encoding. The existing binary example is read; a temporary file demonstrates writing.

In [5]:
with open('example.bin','rb') as file:
    print('existing bytes:',file.read())
with tempfile.TemporaryDirectory() as folder:
    binary=Path(folder)/'demo.bin'
    binary.write_bytes(b'\x00\x01\x02\x03\x04')
    print(binary.read_bytes())

existing bytes: b'\x00\x01\x02\x03\x04'
b'\x00\x01\x02\x03\x04'


## 6. Copy and count text

The original copy workflow reads source then writes destination. `shutil.copy2` is better for large/binary files and preserves metadata. This example copies into a temporary destination.

In [6]:
import shutil
def count_text_file(file_path):
    with open(file_path,encoding='utf-8') as file:
        lines=file.readlines()
    return len(lines),sum(len(line.split()) for line in lines),sum(len(line) for line in lines)
with tempfile.TemporaryDirectory() as folder:
    copied=Path(folder)/'destination.txt'
    shutil.copy2('example.txt',copied)
    print(copied.read_text(encoding='utf-8'))
print(count_text_file('example.txt'))

Hello world
This is a new line 

(2, 7, 32)


## 7. Encoding and newline

Encoding maps characters to bytes. UTF-8 is the portable default. Wrong encoding can raise `UnicodeDecodeError` or produce mojibake. For CSV, pass `newline=""`.

In [7]:
text='Hello नमस्ते 🌍'
encoded=text.encode('utf-8')
print(encoded)
print(encoded.decode('utf-8'))

b'Hello \xe0\xa4\xa8\xe0\xa4\xae\xe0\xa4\xb8\xe0\xa5\x8d\xe0\xa4\xa4\xe0\xa5\x87 \xf0\x9f\x8c\x8d'
Hello नमस्ते 🌍


## Key concepts summary

Use context managers, explicit modes/encoding, streaming for large files, and temporary/backup files before destructive changes.

## Important syntax and quick revision cheat sheet

| Need | Syntax |
|---|---|
| Read text | `open(path,"r",encoding="utf-8")` |
| Write/append | `"w"` / `"a"` |
| Exclusive create | `"x"` |
| Binary | `"rb"`, `"wb"` |
| Pointer | `tell()`, `seek(offset)` |
| Auto-close | `with open(...) as file:` |

## Common mistakes and interview tips

- `w` truncates immediately.
- `writelines` adds no separators.
- Do not mix `str` and `bytes`.
- Avoid `read()` for huge files; stream chunks/lines.
- Check paths and keep backups before overwrite/delete.

**Revision habit:** explain what each line does, predict the result, run it, and test one edge case.